Testando BGE-M3

In [2]:
from FlagEmbedding import BGEM3FlagModel
from FlagEmbedding import FlagReranker
from pymongo import MongoClient
from qdrant_client.models import Prefetch, FusionQuery, Fusion, models
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, SparseVectorParams
from qdrant_client.models import PointStruct, SparseVector
import uuid
import pprint
from bson import ObjectId
import json
from FlagEmbedding import BGEM3FlagModel

Conectar banco

In [3]:
def conectar_banco() -> list[str]:
    client = MongoClient("mongodb://localhost:27017/")
    db = client["TCC"]
    professores = db["Docentes"]
    perfis = list(professores.find({}))
    return perfis 

In [4]:
professores = conectar_banco()

Criação do perfil do professor

In [5]:
def extrair_areas(professor):
    competencias = professor.get("competencias", {})
    areas = competencias.get("areas", [])

    grandes_areas = set()
    areas_atuacao = set()
    subareas = set()

    for item in areas:
        grande_area = item.get("grande_area")
        area = item.get("area")
        subarea = item.get("subarea")

        if grande_area:
            grandes_areas.add(grande_area)

        if area:
            areas_atuacao.add(area)

        if subarea:
            subareas.add(subarea)

    return {
        "grandes_areas": sorted(grandes_areas),
        "areas": sorted(areas_atuacao),
        "subareas": sorted(subareas)
    }

In [6]:
def montar_texto_projeto(projeto):
    titulo = projeto.get("titulo", "")
    descricao = projeto.get("descricao", "")
    area = projeto.get("areaConhecimento", "")
    
    partes = []

    if titulo:
        partes.append(f"Título: {titulo}")

    if descricao:
        partes.append(f"Descrição: {descricao}")

    if area:
        partes.append(f"Área de conhecimento: {area}")

    return "\n".join(partes)

In [ ]:
def montar_componentes_perfil(prof):

    competencias = prof.get("competencias", {})
    iniciacoes = []

    for ic in prof.get("iniciacaoCientifica", []):
        titulo = ic.get("titulo", "")
        resumo = ic.get("resumo", "")
        palavras = ic.get("palavrasChaves", [])

        texto_ic = f"""
                Título: {titulo}

                Resumo: {resumo}

                Palavras-chave: {", ".join(palavras)}
                """.strip()

        iniciacoes.append(texto_ic)

    subareas = [
        a.get("subarea")
        for a in competencias.get("areas", [])
        if a.get("subarea")
    ]
    especialidades = [
        a.get("especialidade")
        for a in competencias.get("areas", [])
        if a.get("especialidade")
    ]
    
    linhas = competencias.get("linhas_pesquisa", [])

    palavras_chave = competencias.get("palavras_chave", [])

    disciplinas = [
        d.get("nome")
        for d in prof.get("disciplinasSigaa", [])
        if d.get("nome")
    ]
    ## bolar o que colocar em cada um desses muita coisa daria para entrar tipo 
    projetos = [
        p.get("titulo")
        for p in prof.get("projetos", [])
        if p.get("titulo")
    ]

    producoes = [
        p.get("titulo")
        for p in prof.get("producoes", {}).get("artigo_periodico", [])
        if p.get("titulo")
    ]

    orientacoes = [
        o.get("titulo")
        for o in prof.get("orientacoes", [])
        if o.get("titulo")
    ]


    return {

        "areas_atuação" : "; ".join(subareas),

        "especialidade" : "; ".join(especialidades),

        "pesquisa": "; ".join(
            linhas +
            palavras_chave
        ),

        "projetos": "; ".join(projetos),

        "producoes": "; ".join(producoes),

        "disciplinas": "; ".join(disciplinas),

        "orientacoes": "; ".join(orientacoes),
        
        "iniciacao_cientifica": "\n\n".join(iniciacoes),
        
    }

BGE-M3

In [ ]:
# bge-m3 vai ficar na mémoria da nossa aplicação
def carregar_bgeM3():
    return BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)''

def rodar_bgeM3(modelo, componentes):
    embeddings = {}
    for nome, texto in componentes.items():
        if not texto or not texto.strip():
            continue

        resultado = modelo.encode(
            texto,
            return_dense=True, 
            return_sparse=True
        )

        embeddings[nome] = {
            "dense": resultado["dense_vecs"],
            "sparse": resultado["lexical_weights"]
        }

    
    return embeddings

def obter_denso(perfis_transformados):
    return perfis_transformados['dense_vecs']
    

In [9]:
modelo = carregar_bgeM3()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Criação do texto por componente

In [10]:
import hashlib

def transformar_bytes(texto : str) -> bytes:
    t = texto
    return t.encode("utf-8")

In [11]:
def gerar_hash(componentes):   
    json_normalizado = json.dumps(
        componentes,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":")
    )
    return hashlib.sha256(transformar_bytes(json_normalizado)).hexdigest()

In [12]:
def criar_perfils_componentes(professores) -> list[str]:
    professores_texto = []
    for professor in professores:
        componentes = montar_componentes_perfil(professor)
        professor = {
            "id" : professor.get("siape"),
            "nome" : professor.get("nome"),
            "componentes" : componentes,
            # pensar hash com componentes separados
            "hash" : {
                nome : gerar_hash(componente)
                for nome, componente in componentes.items()
                
            },
            "metadata" : {
                **extrair_areas(professor),
                "unidade" : professor.get("unidade"),
               
            }
        }
        professores_texto.append(professor)
        
    return professores_texto


In [13]:
textos_professores = {}
professores_transformados = criar_perfils_componentes(professores)

In [18]:
resultados_professores = {}  
for prof in professores_transformados:
    
    componentes = prof.get("componentes", {})
    
    texto_transformado = rodar_bgeM3(modelo,componentes)

    resultados_professores[prof["id"]] = texto_transformado

In [114]:
print(professores_transformados[0])

{'id': '2019312', 'nome': 'Odilon de Oliveira Dutra', 'componentes': {'areas_atuação': 'Circuitos Integrados CMOS; Petróleo e Gas; Circuitos Eletrônicos; ENGENHARIA DE COMPUTAÇÃO; Processamento Digital de Sinais; Linguagens de Programação', 'especialidade': '', 'pesquisa': 'CMOS; Halo Implant; Izhikevich; Translinear; ultra-low-power; ultra-low-voltage; EEG; Electroencephalogram; Low Noise; Low Power; Neural Amplifier; OTA', 'projetos': 'Estudo e desenvolvimento de redes neurais otimizadas para detecção de batimentos ventriculares prematuros (PVC).; Análise de viabilidade de projeto e implementação de processadores RISC-V em FPGA; Implementação de Circuitos Neuromórficos Digitais utilizando a Arquitetura Extreme Learning Machine; Desenvolvimento de um sistema de monitoramento cardíaco em FPGA visando aplicações em IoT; Reconfiguração do processador softcore RVfpga-EL2 para Internet das Coisas.', 'producoes': 'A Low Power R-peak Detector Clocked at Signal Sampling Rate; An Approach to S

In [38]:
resultados_professores['1682528']

{'areas_atuação': {'dense': array([ 0.01897055,  0.03190417, -0.04231234, ...,  0.02840416,
          0.07017302,  0.00774267], shape=(1024,), dtype=float32),
  'sparse': defaultdict(int,
              {'37691': np.float32(0.2537781),
               '7': np.float32(0.085341915),
               '8': np.float32(0.10766861),
               '159415': np.float32(0.23495303),
               '4596': np.float32(0.13149565),
               '74': np.float32(0.13340043),
               '68504': np.float32(0.1546402),
               '14108': np.float32(0.10694977),
               '28': np.float32(0.06812599),
               '151042': np.float32(0.1545577),
               '48': np.float32(0.060858063),
               '83523': np.float32(0.17075962),
               '11': np.float32(0.04594174)})},
 'especialidade': {'dense': array([-0.02739965, -0.00872307, -0.01920665, ...,  0.02963718,
          0.038281  ,  0.00658138], shape=(1024,), dtype=float32),
  'sparse': defaultdict(int,
              {'3

Qdrant

In [25]:
import uuid

def gerar_id(professor_id, tipo):
    identificador = f"{professor_id}:{tipo}"

    return str(uuid.uuid5(uuid.NAMESPACE_DNS, identificador))

In [ ]:
def conectar_qdrant(url):
    return QdrantClient(url=url)

def criar_banco_qdrant(client_qdrant):
    client_qdrant.create_collection(
        collection_name="professores",
        vectors_config={
            "dense": VectorParams(size=1024, distance=Distance.COSINE)},
        sparse_vectors_config={
            "sparse": SparseVectorParams(),
        },
    )

def adicionar_professores_qdrant(client_qdrant, perfis, resultados_professores):
    points = []
    for professor in perfis:
        
        professor_id  = professor["id"]
        
        resultados = resultados_professores[professor_id]

        for componente, resultado in resultados.items():

            sparse = resultado["sparse"]

            sparse_indices = [int(k) for k in sparse.keys()]
            sparse_values = [float(v) for v in sparse.values()]
            
            dense = resultado["dense"]
            
            points.append(
                PointStruct(
                    id=gerar_id(professor, componente),
                    vector={
                        "dense": dense.tolist(),
                        "sparse": SparseVector(indices=sparse_indices, values=sparse_values),
                    },
                    # metadados
                    payload={
                        "nome": professor.get("nome"),
                        "unidade": professor.get("unidade"),
                        "componente": componente,
                        
                    }
                )
            )
            
        client_qdrant.upsert(collection_name="professores", points=points)

# adicionar função para ver se o professor já existe ou se é novo 
def atualizar_perfis(perfis, client_qdrant):

    componentes_para_atualizar = []

    for perfil in perfis:

        professor_id = perfil.get("id")
        hashes = perfil.get("hash", {})
        versao = perfil.get("versao")

        for componente, hash_atual in hashes.items():

            ponto_id = gerar_id(professor_id,componente)
            busca = client_qdrant.retrieve(
                collection_name="professores",
                ids=[ponto_id],
                with_payload=True
            )
            if not busca:
                componentes_para_atualizar.append({
                    "perfil": perfil,
                    "componente": componente
                })
                continue

            payload_qdrant = busca[0].payload

            hash_salvo = payload_qdrant.get("hash")
            versao_salva = payload_qdrant.get("versao")

            # Componente foi alterado
            if (hash_salvo != hash_atual or versao_salva != versao):
                componentes_para_atualizar.append({"perfil": perfil,"componente": componente})

    return componentes_para_atualizar

In [36]:
client_qdrant = conectar_qdrant("http://localhost:6333")

In [66]:
client_qdrant.delete_collection("professores")

True

In [67]:
criar_banco_qdrant(client_qdrant)

In [68]:
adicionar_professores_qdrant(client_qdrant, professores_transformados, resultados_professores)

In [ ]:
atualizar_perfis(client_qdrant,professores_transformados)

In [ ]:
aluno_texto = "Gostaria de desenvolver um projeto com inteligência artificial e programação embarcada"
aluno = modelo.encode(aluno_texto, return_dense=True, return_sparse=True)
embedding_aluno = aluno['dense_vecs']
lexical_aluno = aluno['lexical_weights']

In [151]:
sparse_query = SparseVector(
    indices=[int(k) for k in lexical_aluno.keys()],
    values=[float(v) for v in lexical_aluno.values()],
)

prefetch = [
    Prefetch(
        query=embedding_aluno.tolist(),
        using="dense",
        limit=50
    ),
    Prefetch(
        query=sparse_query,
        using="sparse",
        limit=50
    )
]

results = client_qdrant.query_points(
    collection_name="professores",
    prefetch=prefetch,
    query=FusionQuery(
        fusion=Fusion.RRF
    ),
    limit=50,
    with_payload=True
)

In [152]:
for point in results.points:
    print(
        point.payload["nome"],
        point.payload["componente"],
        point.score
    )

Robson Bauwelz Gonzatti iniciacao_cientifica 0.5416667
Carlos Henrique Valério de Moraes disciplinas 0.5
Rodrigo Maximiano Antunes de Almeida iniciacao_cientifica 0.43333334
Edvard Martins de Oliveira iniciacao_cientifica 0.37333333
Luciano Bertini orientacoes 0.32142857
João Paulo Reus Rodrigues Leite iniciacao_cientifica 0.30882353
João Paulo Reus Rodrigues Leite orientacoes 0.3
Luiz Edival de Souza pesquisa 0.25757575
Bruno Tardiole Kuehne orientacoes 0.20535715
Leonardo Breseghello Zoccal projetos 0.2
Ramon Maia Borges projetos 0.17511521
Rodrigo Maximiano Antunes de Almeida disciplinas 0.16666667
Luciano Bertini pesquisa 0.15625
Rodrigo Maximiano Antunes de Almeida producoes 0.13238771
Breno de Oliveira Renó disciplinas 0.125
Rodrigo Maximiano Antunes de Almeida pesquisa 0.11688312
Rodrigo Maximiano Antunes de Almeida orientacoes 0.11538462
Enzo Seraphim producoes 0.11212122
Carlos Henrique Valério de Moraes pesquisa 0.11111111
Luís Henrique de Carvalho Ferreira orientacoes 0.1109

In [ ]:
professores = defaultdict(list)

for point in results.points:
    professores[point.payload["nome"]].append({
        "nome": point.payload["nome"],
        "componente": point.payload["componente"],
        "score": point.score
    })

ranking = []

for professor_id, componentes in professores.items():

    melhor = max(
        componentes,
        key=lambda x: x["score"]
    )

    ranking.append({
        "professor_id": professor_id,
        "nome": melhor["nome"],
        "score": melhor["score"],
        "componente": melhor["componente"]
    })


ranking.sort(
    key=lambda x: x["score"],
    reverse=True
)

ranking = ranking[:5]

In [144]:
print(ranking)

[{'professor_id': 'Robson Bauwelz Gonzatti', 'nome': 'Robson Bauwelz Gonzatti', 'score': 0.55263156, 'componente': 'iniciacao_cientifica'}, {'professor_id': 'Luís Henrique de Carvalho Ferreira', 'nome': 'Luís Henrique de Carvalho Ferreira', 'score': 0.5, 'componente': 'projetos'}, {'professor_id': 'João Paulo Reus Rodrigues Leite', 'nome': 'João Paulo Reus Rodrigues Leite', 'score': 0.36904764, 'componente': 'iniciacao_cientifica'}, {'professor_id': 'Leonardo Breseghello Zoccal', 'nome': 'Leonardo Breseghello Zoccal', 'score': 0.33333334, 'componente': 'projetos'}, {'professor_id': 'Luciano Bertini', 'nome': 'Luciano Bertini', 'score': 0.32142857, 'componente': 'orientacoes'}]


In [154]:
PESOS = {
    "areas": 0.15,
    "subareas": 0.15,
    "palavras_chave": 0.15,
    "disciplinas": 0.10,
    "projetos": 0.20,
    "iniciacao_cientifica": 0.25
}

In [155]:
def calcular_score(componentes, pesos):

    numerador = 0
    denominador = 0

    for componente in componentes:

        nome = componente["componente"]
        score = componente["score"]

        peso = pesos.get(nome, 0)

        numerador += score * peso
        denominador += peso

    if denominador == 0:
        return 0

    return numerador / denominador

In [156]:
ranking = []

for professor_id, componentes in professores.items():

    score_final = calcular_score(
        componentes,
        PESOS
    )

    ranking.append({
        "nome": componentes[0]["nome"],
        "score": score_final,
        "componentes": componentes
    })

In [157]:
ranking.sort(
    key=lambda x: x["score"],
    reverse=True
)

In [158]:
for professor in ranking[:5]:
    print(
        f"Professor: {professor['nome']}"
    )
    print(
        f"Score: {professor['score']:.4f}"
    )
    print(
        f"Componentes: {professor['componentes']}"
    )
    print("-" * 50)

Professor: Robson Bauwelz Gonzatti
Score: 0.5417
Componentes: [{'nome': 'Robson Bauwelz Gonzatti', 'componente': 'iniciacao_cientifica', 'score': 0.5416667}]
--------------------------------------------------
Professor: Carlos Henrique Valério de Moraes
Score: 0.5000
Componentes: [{'nome': 'Carlos Henrique Valério de Moraes', 'componente': 'disciplinas', 'score': 0.5}, {'nome': 'Carlos Henrique Valério de Moraes', 'componente': 'pesquisa', 'score': 0.11111111}]
--------------------------------------------------
Professor: Edvard Martins de Oliveira
Score: 0.3733
Componentes: [{'nome': 'Edvard Martins de Oliveira', 'componente': 'iniciacao_cientifica', 'score': 0.37333333}]
--------------------------------------------------
Professor: Rodrigo Maximiano Antunes de Almeida
Score: 0.3571
Componentes: [{'nome': 'Rodrigo Maximiano Antunes de Almeida', 'componente': 'iniciacao_cientifica', 'score': 0.43333334}, {'nome': 'Rodrigo Maximiano Antunes de Almeida', 'componente': 'disciplinas', 'sco